# WaterStress Data Pipeline Tutorial

For users who do not intend to use the argument parser, this notebook demonstrates how to use the Water Data Pipeline programmatically within Jupyter (feel free to edit it).

Author: Yanfei Shan, TUM

## Table of Contents
1. [Setup](#1-setup)
2. [Configuration](#2-configuration)
3. [ERA5 Data](#3-era5-data)
4. [MODIS Data](#4-modis-data)
5. [SMIA Data](#5-smia-data)
6. [TWSA Data](#6-twsa-data)
7. [Resampling](#7-resampling)
8. [Merging](#8-merging)
9. [Full Pipeline](#9-full-pipeline)

## 1. Setup

First, let's import the necessary modules and set up the configuration.

In [1]:
# Add project root to path (if running from project directory)
import sys
sys.path.insert(0, '.')

# Import configuration
from config import load_config, create_directories

# Load configuration from config.yaml
config = load_config()

# Create all necessary directories
create_directories(config)

print("Configuration loaded successfully!")
print(f"Working directory: {config.working_dir}")
print(f"Log directory: {config.log_dir}")

Configuration loaded successfully!
Working directory: ./data
Log directory: ./logs


## 2. Configuration

Let's explore the configuration structure.

In [2]:
# View year ranges
print("=== Date Range ===")
print(f"Train: {config.train_start} - {config.train_end}")
print(f"Predict: {config.predict_start} - {config.predict_end}")

# View ERA5 config
print("\n=== ERA5 Config ===")
print(f"Output dir: {config.era5.output_dir}")
print(f"Variables: {config.era5.variables[:3]}... ({len(config.era5.variables)} total)")

# View MODIS config
print("\n=== MODIS Config ===")
print(f"Product: {config.modis.short_name} v{config.modis.version}")
print(f"Selected vars: {config.modis.selected_vars or 'All'}")

# View merge config
print("\n=== Merge Config ===")
print(f"Sources: {config.merge.sources}")
print(f"Extra files: {config.merge.extra_files or 'None'}")

=== Date Range ===
Train: 1990 - 2018
Predict: 2019 - 2025

=== ERA5 Config ===
Output dir: ./data/era5/raw
Variables: ['2m_dewpoint_temperature', '2m_temperature', 'volumetric_soil_water_layer_1']... (12 total)

=== MODIS Config ===
Product: MOD13C2 v061
Selected vars: ['NDVI']

=== Merge Config ===
Sources: {'era5': True, 'modis': True, 'smia': True, 'twsa': True}
Extra files: None


## 3. ERA5 Data

### 3.1 Download ERA5

In [3]:
START_YEAR = 2000
END_YEAR = 2025  # Use single year for testing

In [ ]:
from src.downloaders.era5 import download_era5, process_era5

# Download ERA5 data for specific years
# Note: Requires CDS API to be configured (~/.cdsapirc)

START_YEAR = 2002
END_YEAR = 2002  # Use single year for testing

downloaded_files = download_era5(
    output_dir=config.era5.output_dir,
    variables=config.era5.variables,
    months=config.era5.months,
    start_year=START_YEAR,
    end_year=END_YEAR,
    log_dir=config.log_dir
)
print(f"Downloaded {len(downloaded_files)} files")

### 3.2 Unzip ERA5

In [ ]:
# Unzip downloaded ERA5 files
processed_files = process_era5(
    input_dir=config.era5.output_dir,
    output_dir=config.era5.processed_dir,
    start_year=START_YEAR,
    end_year=END_YEAR,
    log_dir=config.log_dir,
    delete_zip=False  # Set True to delete zip after extraction
)
print(f"Processed {len(processed_files)} files")

## 4. MODIS Data

### 4.1 Download MODIS

In [4]:
from src.downloaders.modis import download_modis, process_modis

# First, login to NASA Earthaccess (run once interactively)
import earthaccess

earthaccess.login()

In [ ]:
# Download MODIS data

downloaded_files = download_modis(
    output_dir=config.modis.raw_dir,
    start_year=START_YEAR,
    end_year=END_YEAR,
    short_name=config.modis.short_name,
    version=config.modis.version,
    log_dir=config.log_dir
)
print(f"Downloaded {len(downloaded_files)} files")

### 4.2 Convert MODIS HDF to NetCDF

In [6]:
# Process MODIS HDF files to NetCDF

processed_files = process_modis(
    input_dir=config.modis.raw_dir,
    output_dir=config.modis.nc_dir,
    start_year=START_YEAR,
    end_year=END_YEAR,
    var_mapping=config.modis.var_mapping,
    selected_vars=config.modis.selected_vars or None,  # None = all variables
    fill_value=config.modis.fill_value,
    log_dir=config.log_dir
)
print(f"Processed {len(processed_files)} files")

2026-03-05 14:04:00,598 - modis_process - INFO - Processing selected variables: ['NDVI']
2026-03-05 14:04:00,600 - modis_process - INFO - Processing year: 2000
2026-03-05 14:04:00,601 - modis_process - INFO - Processing: MOD13C2.A2000032.061.2020042083108.hdf
2026-03-05 14:04:00,744 - modis_process - INFO -   Reading: CMG 0.05 Deg Monthly NDVI -> NDVI
2026-03-05 14:04:01,083 - modis_process - INFO -     SDS attrs: fill=-3000, valid_range=[-2000, 10000], scale_factor=10000.0, add_offset=0.0
2026-03-05 14:04:01,376 - modis_process - INFO -     After QC: range [-0.2000, 0.9991], NaN ratio: 76.40%
2026-03-05 14:04:01,389 - modis_process - INFO - Processing: MOD13C2.A2000061.061.2020041172815.hdf
2026-03-05 14:04:01,493 - modis_process - INFO -   Reading: CMG 0.05 Deg Monthly NDVI -> NDVI
2026-03-05 14:04:01,802 - modis_process - INFO -     SDS attrs: fill=-3000, valid_range=[-2000, 10000], scale_factor=10000.0, add_offset=0.0
2026-03-05 14:04:02,050 - modis_process - INFO -     After QC: r

## 5. SMIA Data

Convert SMIA TIFF files to NetCDF.

In [ ]:
from src.processors.smia import process_smia

# Process SMIA TIFF files
# Note: TIFF files should be organized in year folders

processed_files = process_smia(
    input_dir=config.smia.input_dir,
    output_dir=config.smia.output_dir,
    days=config.smia.days,
    start_year=START_YEAR,
    end_year=END_YEAR,
    log_dir=config.log_dir
)
print(f"Processed {len(processed_files)} files")

## 6. TWSA Data

Process TWSA data with automatic month filling.

In [ ]:
from src.processors.twsa import process_twsa

# Process TWSA data
# - Automatically finds files with flexible naming patterns
# - Fills missing months using forward-fill

processed_files = process_twsa(
    input_dir=config.twsa.input_dir,
    output_dir=config.twsa.output_dir,
    start_year=2020,
    end_year=2025,
    reference_grid_path=config.resampling.reference_grid,
    fill_missing_months=True,  # Set False to disable month filling
    log_dir=config.log_dir
)
print(f"Processed {len(processed_files)} files")

## 7. Resampling

Resample all data sources to a common reference grid.

In [ ]:
from src.processors.resampling import resample_era5, resample_modis, resample_smia

# Resample ERA5
resample_era5(
    input_dir=config.era5.processed_dir,
    output_dir=config.era5.resampled_dir,
    reference_grid_path=config.resampling.reference_grid,
    start_year=START_YEAR,
    end_year=END_YEAR,
    method=config.resampling.method,
    log_dir=config.log_dir
)

In [10]:
# Resample MODIS
resample_modis(
    input_dir=config.modis.nc_dir,
    output_dir=config.modis.resampled_dir,
    reference_grid_path=config.resampling.reference_grid,
    start_year=START_YEAR,
    end_year=END_YEAR,
    method=config.resampling.method,
    log_dir=config.log_dir
)

2026-03-04 12:26:34,410 - modis_resample - INFO - Loaded reference grid: 4320 lon x 2160 lat


Resampling MODIS:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-04 12:26:34,426 - modis_resample - INFO - Loading: ./data/modis/nc/2002.nc
2026-03-04 12:26:34,471 - modis_resample - INFO - Interpolating to target resolution...
2026-03-04 12:28:12,475 - modis_resample - INFO - Saving to ./data/modis/resampled/2002.nc...
2026-03-04 12:28:46,689 - modis_resample - INFO - Done: ./data/modis/resampled/2002.nc with shape Frozen({'time': 12, 'lat': 2160, 'lon': 4320})


Resampling MODIS: 100%|██████████| 1/1 [02:12<00:00, 132.37s/it]


['./data/modis/resampled/2002.nc']

In [ ]:
# Resample SMIA
resample_smia(
    input_dir=config.smia.output_dir,
    output_dir=config.smia.resampled_dir,
    reference_grid_path=config.resampling.reference_grid,
    start_year=START_YEAR,
    end_year=END_YEAR,
    method=config.resampling.method,
    log_dir=config.log_dir
)

## 8. Merging

Merge all data sources into single NetCDF files per year.

In [ ]:
from src.processors.merger import merge_data

# Merge all enabled sources
merged_files = merge_data(
    era5_dir=config.era5.resampled_dir,
    modis_dir=config.modis.resampled_dir,
    smia_dir=config.smia.resampled_dir,
    twsa_dir=config.twsa.output_dir,
    output_dir=config.merge.output_dir,
    start_year=START_YEAR,
    end_year=END_YEAR,
    sources_enabled=config.merge.sources,
    extra_files=config.merge.extra_files or None,
    compression_level=config.merge.compression_level,
    require_all=False,
    log_dir=config.log_dir
)
print(f"Merged {len(merged_files)} files")

### 8.1 Custom Merge (Selective Sources)

In [ ]:
# Example: Merge only ERA5 and MODIS

custom_sources = {
    'era5': True,
    'modis': True,
    'smia': False,
    'twsa': False
}

merged_files = merge_data(
    era5_dir=config.era5.resampled_dir,
    modis_dir=config.modis.resampled_dir,
    smia_dir=config.smia.resampled_dir,
    twsa_dir=config.twsa.output_dir,
    output_dir="./data/merged_era5_modis",
    start_year=START_YEAR,
    end_year=END_YEAR,
    sources_enabled=custom_sources,
    log_dir=config.log_dir
)

### 8.2 Merge with Extra Files

In [ ]:
# Example: Merge with additional data files

extra_files = [
    "./data/extra/elevation.nc",
    "./data/extra/land_cover.nc"
]

merged_files = merge_data(
    era5_dir=config.era5.resampled_dir,
    modis_dir=config.modis.resampled_dir,
    smia_dir=config.smia.resampled_dir,
    twsa_dir=config.twsa.output_dir,
    output_dir="./data/merged_with_extra",
    start_year=START_YEAR,
    end_year=END_YEAR,
    sources_enabled=config.merge.sources,
    extra_files=extra_files,
    log_dir=config.log_dir
)

## 9. Full Pipeline

Run the complete pipeline from start to finish.

In [ ]:
def run_full_pipeline(start_year, end_year, config):
    """
    Run the complete data processing pipeline.
    
    Args:
        start_year: First year to process
        end_year: Last year to process
        config: Configuration object
    """
    from src.downloaders.era5 import download_era5, process_era5
    from src.downloaders.modis import download_modis, process_modis
    from src.processors.smia import process_smia
    from src.processors.twsa import process_twsa
    from src.processors.resampling import resample_era5, resample_modis, resample_smia
    from src.processors.merger import merge_data
    
    print("=" * 60)
    print(f"Running Full Pipeline: {start_year} - {end_year}")
    print("=" * 60)
    
    # Step 1: ERA5
    print("\n[1/10] Downloading ERA5...")
    download_era5(
        output_dir=config.era5.output_dir,
        variables=config.era5.variables,
        months=config.era5.months,
        start_year=start_year,
        end_year=end_year,
        log_dir=config.log_dir
    )
    
    print("\n[2/10] Unzipping ERA5...")
    process_era5(
        input_dir=config.era5.output_dir,
        output_dir=config.era5.processed_dir,
        start_year=start_year,
        end_year=end_year,
        log_dir=config.log_dir
    )
    
    # Step 2: MODIS
    print("\n[3/10] Downloading MODIS...")
    download_modis(
        output_dir=config.modis.raw_dir,
        start_year=max(start_year, 2000),
        end_year=end_year,
        log_dir=config.log_dir
    )
    
    print("\n[4/10] Converting MODIS HDF to NC...")
    process_modis(
        input_dir=config.modis.raw_dir,
        output_dir=config.modis.nc_dir,
        start_year=max(start_year, 2000),
        end_year=end_year,
        log_dir=config.log_dir
    )
    
    # Step 3: SMIA
    print("\n[5/10] Processing SMIA...")
    process_smia(
        input_dir=config.smia.input_dir,
        output_dir=config.smia.output_dir,
        days=config.smia.days,
        start_year=start_year,
        end_year=end_year,
        log_dir=config.log_dir
    )
    
    # Step 4: TWSA
    print("\n[6/10] Processing TWSA...")
    process_twsa(
        input_dir=config.twsa.input_dir,
        output_dir=config.twsa.output_dir,
        start_year=max(start_year, 2002),
        end_year=end_year,
        reference_grid_path=config.resampling.reference_grid,
        log_dir=config.log_dir
    )
    
    # Step 5: Resampling
    print("\n[7/10] Resampling ERA5...")
    resample_era5(
        input_dir=config.era5.processed_dir,
        output_dir=config.era5.resampled_dir,
        reference_grid_path=config.resampling.reference_grid,
        start_year=start_year,
        end_year=end_year,
        log_dir=config.log_dir
    )
    
    print("\n[8/10] Resampling MODIS...")
    resample_modis(
        input_dir=config.modis.nc_dir,
        output_dir=config.modis.resampled_dir,
        reference_grid_path=config.resampling.reference_grid,
        start_year=max(start_year, 2000),
        end_year=end_year,
        log_dir=config.log_dir
    )
    
    print("\n[9/10] Resampling SMIA...")
    resample_smia(
        input_dir=config.smia.output_dir,
        output_dir=config.smia.resampled_dir,
        reference_grid_path=config.resampling.reference_grid,
        start_year=start_year,
        end_year=end_year,
        log_dir=config.log_dir
    )
    
    # Step 6: Merge
    print("\n[10/10] Merging data...")
    merge_data(
        era5_dir=config.era5.resampled_dir,
        modis_dir=config.modis.resampled_dir,
        smia_dir=config.smia.resampled_dir,
        twsa_dir=config.twsa.output_dir,
        output_dir=config.merge.output_dir,
        start_year=start_year,
        end_year=end_year,
        sources_enabled=config.merge.sources,
        log_dir=config.log_dir
    )
    
    print("\n" + "=" * 60)
    print("Pipeline completed!")
    print("=" * 60)

In [ ]:
# Run the full pipeline
run_full_pipeline(2020, 2024, config)

## Utility Functions

Some helpful utility functions for data inspection.

In [ ]:
import xarray as xr
from src.utils import check_dataset_variables, get_size

# Check file size
# print(get_size("./data/merged/2020_merged.nc"))

# Load and inspect a dataset
# ds = xr.open_dataset("./data/merged/2020_merged.nc")
# print(ds)
# print(f"\nVariables: {list(ds.data_vars)}")
# print(f"Dimensions: {dict(ds.dims)}")

In [ ]:
# Check if all variables can be loaded
# ds = xr.open_dataset("./data/merged/2020_merged.nc")
# check_dataset_variables(ds)

## CLI Equivalent Commands

The same operations can be performed from the command line:

```bash
# ERA5
python3 main.py era5-download --start-year 2025 --end-year 2025
python3 main.py era5-unzip --start-year 2025 --end-year 2025
python3 main.py resample-era5 --start-year 2025 --end-year 2025

# MODIS
python3 main.py modis-download --start-year 2025 --end-year 2025
python3 main.py modis-hdftonc --start-year 2025 --end-year 2025
python3 main.py resample-modis --start-year 2025 --end-year 2025
python3 main.py modis-landcover --start-year 2020 --end-year 2023
python3 main.py modis-landuse --start-year 2020 --end-year 2023
python3 main.py resample-modis-landcover --start-year 2020 --end-year 2023
python3 main.py resample-modis-landuse --start-year 2020 --end-year 2023

# SMIA
python3 main.py smia-tifftonc --start-year 2025 --end-year 2025
python3 main.py resample-smia --start-year 2025 --end-year 2025

# TWSA
python3 main.py twsa-resample --start-year 2025 --end-year 2025

# Merge
python3 main.py merge --start-year 2025 --end-year 2025

# Full pipeline
python3 main.py full-pipeline --start-year 2025 --end-year 2025 --continue-on-error
```